# Week 14: LLM Fine-Tuning & Quantization
## Notebook 07: BitNet b1.58: Ternary Weight Quantization {-1, 0, 1}

### 1. Conceptual Intuition & Theoretical Background
In standard LLM inference, matrix multiplications ($Y = W X$) between FP16 weights and FP16 activations consume enormous GPU energy and memory bandwidth.

**BitNet b1.58** constrains every weight parameter to ternary values:
$$W_{ij} \in \{-1, 0, 1\}$$
Representable in just $\log_2(3) \approx 1.58$ bits!

**BitLinear Layer Mechanics:**
1. **Absmean Weight Quantization:**
$$\gamma = \frac{1}{nm} \sum_{i,j} |W_{ij}|$$
$$\tilde{W} = \text{RoundClip}\left( \frac{W}{\gamma + \epsilon},\ -1,\ 1 \right)$$

2. **Absmax Activation Quantization (to 8-bit):**
$$\eta = \max_{i,j} |X_{ij}|$$
$$\tilde{X} = \text{Clip}\left( \frac{X}{\eta} \cdot 127,\ -128,\ 127 \right)$$

3. **Multiplication-Free Matrix Multiplication:**
Matrix multiplication simplifies entirely into **integer additions and subtractions**:
$$Y = \tilde{W} \tilde{X}$$
Eliminating energy-intensive floating-point multiply-accumulate (MAC) units.

---

### 2. Scratch Python Implementation: BitLinear with Straight-Through Estimator

In [ ]:
import numpy as np

def round_clip(x, min_val=-1, max_val=1):
    return np.clip(np.round(x), min_val, max_val)

class BitLinearScratch:
    def __init__(self, in_features, out_features):
        self.in_features = in_features
        self.out_features = out_features
        # Latent continuous full-precision weights
        np.random.seed(42)
        self.weight = np.random.normal(0, 0.05, (out_features, in_features))
        
    def quantize_weights(self):
        # Absmean scaling factor gamma
        gamma = np.mean(np.abs(self.weight)) + 1e-7
        # Ternary quantized weights in {-1, 0, 1}
        w_quant = round_clip(self.weight / gamma, -1, 1)
        return w_quant, gamma
        
    def quantize_activations(self, x):
        # Scale to 8-bit signed integer [-128, 127]
        eta = np.max(np.abs(x)) + 1e-7
        x_quant = np.clip(np.round((x / eta) * 127.0), -128, 127)
        return x_quant, eta
        
    def forward(self, x):
        w_quant, gamma = self.quantize_weights()
        x_quant, eta = self.quantize_activations(x)
        
        # Computation: Pure Integer Matrix Multiplication!
        # Every element is either added, subtracted, or skipped (0)
        out_int = np.dot(x_quant, w_quant.T)
        
        # Dequantize back to floating point scale
        out = out_int * (gamma * eta / 127.0)
        return out, w_quant

layer = BitLinearScratch(in_features=8, out_features=4)
x_in = np.random.randn(2, 8)

y_out, ternary_w = layer.forward(x_in)

unique_weights = np.unique(ternary_w)
print("--- BitNet b1.58 Ternary Quantization ---")
print(f"Unique Weight Values:       {unique_weights.tolist()} (Strictly -1, 0, 1)")
print(f"Weight Sparsity (zeros %):   {np.mean(ternary_w == 0) * 100:.1f}%")
print(f"Forward Output Shape:       {y_out.shape}")
print("Floating-point multiplications eliminated from core matrix kernel!")

### 3. Step-by-Step Code Walkthrough
1. **Absmean Normalization**: Centers continuous latent weights around the unit boundary before rounding.
2. **Straight-Through Estimator (STE)**: During backward pass, gradients bypass the discrete rounding step ($\frac{\partial \tilde{W}}{\partial W} \approx 1$).
3. **Hardware Efficiency**: Enables high-throughput silicon designs utilizing simple adder trees.

---

### 4. Industry Applications & Trade-offs
- **Ultra-Low Power Edge Silicon:** Running 7B parameter models on mobile phones with minimal battery consumption.
- **Trade-off:** Training requires full-precision latent weights and specialized training kernels.

---

### 5. Top Interview Questions & Answers
**Q1: How does BitNet 1.58b eliminate floating-point multiplications?**
> **Answer:** Because weights are constrained to $\{-1, 0, 1\}$, multiplying an activation by a weight consists of negation ($-1$), zeroing ($0$), or identity ($+1$). The dot product becomes pure integer summation.

**Q2: What is the purpose of the Straight-Through Estimator (STE)?**
> **Answer:** The rounding function has a derivative of zero almost everywhere and is discontinuous at half-integers, which would zero out backpropagated gradients. The STE approximates $\frac{\partial \text{round}(x)}{\partial x} = 1$ during the backward pass.